---
# COSC2673 | COSC2793 (Computational) Machine Learning
## Week 4 Lab: **Training a Regression Model**
---

# Introduction

In the previous weeks, we learned how to read data, perform exploratory data analysis (EDA), and prepare datasets for training. The next step in a typical machine learning pipeline is to train a model and examine the results.

This lab assumes that you have completed prior labs. If you have not, please do so before attempting this lab.

## Objective

- Continue to familiarise yourself with Python and the common machine learning packages.
- Train a regression model and complete the introduction to the model development pipeline.
- Analyse a trained model and its predictions.

## Dataset

This lab works with two regression datasets. The first concerns house prices and associated attributes; the task is to predict the median value (`MEDV`) from various features. The second dataset involves daily bike‑sharing counts in Washington D.C., with predictors such as season, weekday and weather.

The data files (`housing.data.csv` and `bikeShareDay.csv`) are available on Canvas.

Before you begin, make sure the two data directories have been copied into your notebook folder. On a local machine, simply copy the directories.

# Problem Formulation

To build a model, we must first formulate the task in machine learning terms. For the Boston housing dataset, the goal is to predict the house price (`MEDV`) from attributes of the house and neighbourhood.

- **Is there a pattern?**  Examine the data (and your EDA from previous weeks) to see if the features appear correlated with price.

- **Task category:** supervised, multivariate regression.

Since both the inputs and the target are available, we can train directly. The next decision is to choose an appropriate performance measure.

- **Performance measure:** we will use the coefficient of determination \(R^2\). This metric is intuitive compared to mean squared error because it compares the model's performance to a trivial predictor that always outputs the mean of the training target. Use insights from the EDA (e.g. presence of outliers) when choosing a measure.

# Load the dataset and preprocess

Use the techniques from the last two weeks to load the Boston housing data and preprocess it (train/test splitting and normalization).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

bostonHouseFrame = pd.read_csv('./BostonHousingPrice/housing.data.csv', delimiter='\s+')
bostonHouseFrame.head()

We separate the features (attributes) from the target variable.

In [ ]:
bostonHouse_X = bostonHouseFrame.drop(['MEDV'], axis=1)
bostonHouse_y = bostonHouseFrame['MEDV']

Reserve a portion of the data for testing.

In [ ]:
from sklearn.model_selection import train_test_split

with pd.option_context('mode.chained_assignment', None):
    bostonHouse_X_train, bostonHouse_X_test, bostonHouse_y_train, bostonHouse_y_test = train_test_split(bostonHouse_X, bostonHouse_y, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
bostonHouse_X_test

Perform feature scaling.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer

logNorm_attributes = ['CRIM','NOX','AGE','DIS','LSTAT']
minmax_attributes = list(set(bostonHouse_X.columns).difference(set(logNorm_attributes)))

bostonHouse_X_train_scaled = bostonHouse_X_train.copy().astype('object')
bostonHouse_X_test_scaled = bostonHouse_X_test.copy().astype('object')

minmaxscaler = MinMaxScaler().fit(bostonHouse_X_train_scaled.loc[:, minmax_attributes])
bostonHouse_X_train_scaled.loc[:, minmax_attributes] = minmaxscaler.transform(bostonHouse_X_train_scaled.loc[:, minmax_attributes])
bostonHouse_X_test_scaled.loc[:, minmax_attributes] = minmaxscaler.transform(bostonHouse_X_test_scaled.loc[:, minmax_attributes])

powertransformer = PowerTransformer(method='yeo-johnson', standardize=False).fit(bostonHouse_X_train.loc[:, logNorm_attributes])
bostonHouse_X_train_scaled.loc[:, logNorm_attributes] = powertransformer.transform(bostonHouse_X_train.loc[:, logNorm_attributes])
bostonHouse_X_test_scaled.loc[:, logNorm_attributes] = powertransformer.transform(bostonHouse_X_test.loc[:, logNorm_attributes])

minmaxscaler_pt = MinMaxScaler().fit(bostonHouse_X_train_scaled.loc[:, logNorm_attributes])
bostonHouse_X_train_scaled.loc[:, logNorm_attributes] = minmaxscaler_pt.transform(bostonHouse_X_train_scaled.loc[:, logNorm_attributes])
bostonHouse_X_test_scaled.loc[:, logNorm_attributes] = minmaxscaler_pt.transform(bostonHouse_X_test_scaled.loc[:, logNorm_attributes])

Plot the feature distributions to verify that scaling worked correctly.

In [ ]:
plt.figure(figsize=(20,20))
for i, col in enumerate(bostonHouse_X_train_scaled.columns):
    plt.subplot(4,5,i+1)
    plt.hist(bostonHouse_X_train_scaled[col], alpha=0.3, color='b', density=True)
    plt.hist(bostonHouse_X_test_scaled[col], alpha=0.3, color='r', density=True)
    plt.title(col)
    plt.xticks(rotation='vertical')

With the data prepared, we can now build regression models.

# Building a Regression Model

We have already determined that the task is regression and that we will use \(R^2\) as the performance measure. Before trying complex models it is good to establish a simple baseline.

**Baseline model:** a linear regression model.

- The EDA showed that some features (e.g. `'RM'`) exhibit a linear relationship with price.
- Linear regression is simple and interpretable, making it a natural starting point.

Let's train a linear regression model on the scaled training set using
`sklearn.linear_model.LinearRegression`.  Consult the [documentation]
(https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
for parameter details.

In [ ]:
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression().fit(bostonHouse_X_train_scaled, bostonHouse_y_train)

`LinearRegression.fit()` performs the optimisation under the hood and returns
the fitted parameters (weights).

In [ ]:
print("Parameters of the Linear model: ", model_lr.coef_)
print("Intercept of the Linear model: ", model_lr.intercept_)

We can now use the trained model to make predictions on unseen data. This
procedure is called *prediction*.

In [ ]:
bostonHouse_y_test_pred = model_lr.predict(bostonHouse_X_test_scaled)

The array `bostonHouse_y_test_pred` holds the model’s predictions.  How do
these compare to the actual house prices in the test set?  A scatter plot helps
us visualise the relationship.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(bostonHouse_y_test, bostonHouse_y_test_pred, s=25, zorder=10)

lims = [
    np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
    np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

ax.plot(lims, lims, 'k--', alpha=0.75, zorder=0)
ax.plot(lims, [np.mean(bostonHouse_y_train),]*2, 'r--', alpha=0.75, zorder=0)
ax.set_aspect('equal')
ax.set_xlim(lims)
ax.set_ylim(lims)

plt.xlabel('Actual House Price')
plt.ylabel('Predicted House Price')

plt.show()

**Observations:**  
- The model predicts housing prices reasonably well on unseen examples.  
- It outperforms the baseline that always predicts the training mean (red line).

Let's compute a quantitative performance metric.

In [ ]:
from sklearn.metrics import r2_score

r2_lr = r2_score(bostonHouse_y_test, bostonHouse_y_test_pred)
print('The R^2 score for the linier regression model is: {:.3f}'.format(r2_lr))


**Questions:**  
- How would you interpret the \(R^2\) score? Discuss your interpretation with your tutor.  
- Does feature scaling provide an advantage? As a quick experiment, fit the same linear model to the unscaled features and compare the result.

In [ ]:
model_us_lr = LinearRegression().fit(bostonHouse_X_train, bostonHouse_y_train)
bostonHouse_y_test_us_pred = model_us_lr.predict(bostonHouse_X_test)

r2_us_lr = r2_score(bostonHouse_y_test, bostonHouse_y_test_us_pred)
print('The R^2 score for the linier regression model (without feature scaling) is: {:.3f}'.format(r2_us_lr))

Next week we'll delve deeper into model building.  For now, assume that the
linear regression model's performance is satisfactory.

# Explanatory Model Analysis

We now have a model that can predict house prices reasonably well, but it is still a black box: we supply features and receive a prediction.  The test set results give empirical evidence that the model generalises, but is that evidence sufficient? Has the model learned a sensible pattern or merely exploited quirks in this dataset?

To investigate, we need to inspect the model more closely. Although we are not yet covering all the theory, the following sections introduce a few basic techniques for model examination.

## Residual plots

We have already used one technique (prediction vs actual plot). Another common tool that can be used to explore a model is to plot the residuals (deviation from the actual value). 

In [ ]:
fig, ax = plt.subplots()
ax.scatter(bostonHouse_y_test, bostonHouse_y_test-bostonHouse_y_test_pred, s=25, zorder=10)

xlims = ax.get_xlim()
ax.plot(xlims, [0.0,]*2, 'k--', alpha=0.75, zorder=0)
ax.set_xlim(xlims)

plt.xlabel('Actual House Price')
plt.ylabel('Residual')

plt.show()

The test set (or even better a validation set) should be used for this analysis. Train set may be over-fitted to data.

**Observations:**  
For a good model, residuals should behave randomly and be centred around zero.  Systematic patterns (e.g. curvature) or non‑uniform variance suggest model issues such as nonlinearity or heteroscedasticity.

There are several additional diagnostic plots:

- [Understanding Diagnostic Plots for Linear Regression Analysis](https://data.library.virginia.edu/diagnostic-plots/)
- [Creating Diagnostic Plots in Python](https://robert-alvarez.github.io/2018-06-04-diagnostic_plots/)  
  (the examples use `statsmodels`; you can adapt them for scikit‑learn).

## Feature importance

For simple models like linear regression, we can estimate each feature's contribution by examining its coefficient.

In [ ]:
coefs = pd.DataFrame(
    model_lr.coef_  * bostonHouse_X_train_scaled.std(axis=0),
    columns=['Coefficient importance'], index=bostonHouse_X_train_scaled.columns
)
coefs.sort_values(by=['Coefficient importance']).plot(kind='barh', figsize=(9, 7))
plt.title('Ridge model, small regularization')
plt.axvline(x=0, color='.5')
plt.subplots_adjust(left=.3)
plt.show()

Coefficients must be scaled to a common unit to reflect importance – dividing by the feature's standard deviation is a simple proxy.

Correlated features can make coefficients unstable and their effects difficult to disentangle.

**Observations:**  
- The number of rooms (`RM`) and highway accessibility (`RAD`) have the largest positive influence on predicted price.  
- High `LSTAT`, long distances to employment centres (`DIS`), high NOX concentration, and high property tax rate (`TAX`) decrease prices.  
- The negative coefficient on `TAX` is suspicious; it likely reflects industrial areas with high taxes rather than a causal effect. In feature selection we might drop `TAX` for this reason.  
- `ZN` and `INDUS` appear unimportant for predicting price.

An alternative, model‑agnostic measure of feature importance is **permutation
importance**, which is available for any fitted estimator.  See the
[scikit‑learn documentation](https://scikit-learn.org/stable/modules/permutation_importance.html)
for details.

In [ ]:
from sklearn.inspection import permutation_importance

r = permutation_importance(model_lr, bostonHouse_X_test_scaled, bostonHouse_y_test, n_repeats=30)
inx = np.argsort(r.importances_mean)

plt.barh(bostonHouse_X_test_scaled.columns[inx], r.importances_mean[inx])
plt.xticks(rotation='vertical')
plt.show()

**Question:** Is permutation importance the same as plotting regression coefficients?

Many additional techniques exist for model inspection; we'll explore more
throughout the course.

# Exercise: Analyse the Bike‑Share Data

**Task:** Develop a linear regression model for the bike‑share dataset.
Follow the same steps as for the house‑price data.

Answer and discuss the following questions with your tutor:

- Which performance measures are appropriate for this task?
- What would be a suitable baseline model?
- What is the baseline model’s performance, and is it adequate?
- Can you trust your model?

Don't wait for solutions to be released (they will not be!) — attempt the exercise yourself.